# Tri-subject kinship verification: Kaggle runner

This notebook only **runs** the project; all code lives in the Git repo.

**Before running**
1. Settings (right panel): **Accelerator = GPU T4 x2 or P100**, **Internet = On**.
2. **Add Input** → your private dataset `fiw-fids` (or the competition *Recognizing Faces in the Wild*).
3. Edit `REPO_URL` in the next cell.
4. Best: **Save Version → Save & Run All** so it runs in the background and keeps `/kaggle/working` as output.

**Reusing an earlier run (saves GPU time):** after a saved run, attach that run's output as an input
(Add Input → Your Work → this notebook). With `CACHE_FROM = "auto"` its embeddings are found and copied,
and face extraction is skipped.

In [ ]:
# ---- settings -------------------------------------------------------------
REPO_URL = "https://github.com/samikhan05/kinship-verification.git"
BRANCH = "main"
CODE_DATASET = None    # alternative to GitHub: name of a Kaggle dataset holding the repo zip, e.g. "kinship-code"
CACHE_FROM = "auto"    # reuse an attached earlier run's embeddings ("auto" finds it); None = never
WORK = "/kaggle/working/work"
OUT = "/kaggle/working/outputs"

## 1. Get the code

In [ ]:
import glob, os, shutil, subprocess
REPO = "/kaggle/working/kinship-verification"
if CODE_DATASET:
    cfg_file = glob.glob(f"/kaggle/input/{CODE_DATASET}/**/configs/default.yaml", recursive=True)[0]
    shutil.copytree(os.path.dirname(os.path.dirname(cfg_file)), REPO, dirs_exist_ok=True)
elif os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
!git log -1 --oneline 2>/dev/null || echo "(not a git checkout)"

## 2. Install packages
The exact versions and the reasons for them are in `scripts/kaggle_install.sh` (kept in the repo, so fixes arrive with the code).

In [ ]:
!bash scripts/kaggle_install.sh

## 3. Check the GPU (ONNX Runtime must list `CUDAExecutionProvider`)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, onnxruntime as ort, insightface
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())
print("insightface", insightface.__version__)

## 4. Find the data and set paths

In [ ]:
import sys
sys.path.insert(0, f"{REPO}/src")
from kinship.data import find_data_root
from kinship.utils import restore_cached_work
DATA_ROOT = find_data_root()     # fast search of /kaggle/input that never lists the image files
print("Dataset found at:", DATA_ROOT)
restored = restore_cached_work(WORK, CACHE_FROM)
print("Reusing cached embeddings from:", restored or "nothing found - they will be computed")
ARGS = f"--data_root {DATA_ROOT} --work_dir {WORK} --output_dir {OUT}"
print(ARGS)

## 5. Unit tests (CPU, synthetic data, ~10 s)

In [ ]:
!python -m pytest -q

## 6. Step 0: data inspection → `outputs/data_summary.md`

In [ ]:
!python scripts/inspect_data.py {ARGS}

## 7. Face detection + alignment + ArcFace embeddings + gender/age (cached, resumable)
Watch the first log lines: they must say the models run on `CUDAExecutionProvider`.

In [ ]:
!python scripts/extract_embeddings.py {ARGS}

## 8. Family-disjoint splits, triplets, negatives → `work/splits/`, `outputs/triplet_check.jpg`

In [ ]:
!python scripts/prepare.py {ARGS} --force   # cheap and deterministic: always rebuilt
from IPython.display import Image, display
display(Image(f"{OUT}/triplet_check.jpg", width=420))

## 9. Baselines E0 (cosine) and E1 (logistic regression / SVM)

In [ ]:
!python scripts/evaluate.py {ARGS} --exp E0 E1

## 10. FaceNet embeddings (for E2b now and fine-tuning in E3)
MTCNN 160 px crops + InceptionResnetV1 (VGGFace2). Cached and resumable like the ArcFace step;
skipped when an attached earlier run already has them.

In [ ]:
!python scripts/extract_embeddings.py {ARGS} --aligner facenet

## 11. E0 / E1 on FaceNet embeddings (reference rows for E2b)

In [ ]:
!python scripts/evaluate.py {ARGS} --embeddings facenet --exp E0 E1

## 12. E2: attention-gated tri-subject head on frozen ArcFace (main model)
Trains in minutes on the cached embeddings. The log shows validation AUC per epoch; the last line
reports the attention analysis (does alpha follow the parent the child looks more like?).

In [ ]:
!python scripts/train_head.py {ARGS} --embeddings arcface --name E2

## 13. E2-hardneg: same head, half of the training negatives swap one parent

In [ ]:
!python scripts/train_head.py {ARGS} --embeddings arcface --name E2-hardneg --set train.head.hard_negative_fraction=0.5

## 14. E2b: same head on frozen FaceNet embeddings

In [ ]:
!python scripts/train_head.py {ARGS} --embeddings facenet --name E2b

## 15. Report → `outputs/results.md`

In [ ]:
!python scripts/make_report.py {ARGS} > /dev/null
from IPython.display import Markdown
display(Markdown(open(f"{OUT}/results.md", encoding="utf-8").read()))

## 16. Package the outputs for download

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip outputs && ls -lh outputs.zip